# 🛒 Análise Exploratória — Vendas Amazon

**Objetivo:** Explorar o comportamento de vendas da plataforma Amazon, identificando padrões de preço, volume por categoria e distribuição geográfica dos clientes.

**Dataset:** `amazon_df.csv`  
**Ferramentas:** Python · pandas · matplotlib · seaborn  
**Autor:** Darlan

---

## Perguntas de negócio:
1. Como os preços se distribuem? Existe concentração em alguma faixa?
2. Quais categorias vendem mais em volume?
3. Qual categoria gera mais receita (preço × quantidade)?
4. Como as regiões se comparam em volume de compras?
5. Existe correlação entre preço e quantidade vendida?

## 1. Importações e Carregamento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

amazon_df = pd.read_csv('data base/amazon_df.csv')

print(f'Shape: {amazon_df.shape}')
print(f'Colunas: {list(amazon_df.columns)}')
amazon_df.head()

## 2. Visão Geral — Qualidade dos Dados

> Antes de qualquer análise, verificamos nulos, tipos e inconsistências.

In [ ]:
print('=== Tipos de dados ===')
print(amazon_df.dtypes)

print('\n=== Valores nulos por coluna ===')
nulos = amazon_df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Nenhum valor nulo encontrado ✅')

print('\n=== Estatísticas descritivas ===')
amazon_df[['price', 'quantity_sold']].describe().round(2)

## 3. Distribuição de Preços

> Entender como os preços se distribuem revela o posicionamento dos produtos — se a plataforma é dominada por itens baratos, premium ou equilibrada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de preços
axes[0].hist(amazon_df['price'], bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(amazon_df['price'].mean(), color='tomato', linestyle='--', label=f'Média: ${amazon_df["price"].mean():.2f}')
axes[0].axvline(amazon_df['price'].median(), color='orange', linestyle='--', label=f'Mediana: ${amazon_df["price"].median():.2f}')
axes[0].set_title('Distribuição de Preços', fontweight='bold')
axes[0].set_xlabel('Preço (US$)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Boxplot de preços
axes[1].boxplot(amazon_df['price'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='tomato', linewidth=2))
axes[1].set_title('Boxplot de Preços', fontweight='bold')
axes[1].set_ylabel('Preço (US$)')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

# Faixas de preço
bins = [0, 20, 50, 100, 200, float('inf')]
labels = ['Até $20', '$20-50', '$50-100', '$100-200', 'Acima de $200']
amazon_df['faixa_preco'] = pd.cut(amazon_df['price'], bins=bins, labels=labels)
print('\n=== Distribuição por Faixa de Preço ===')
print(amazon_df['faixa_preco'].value_counts().sort_index())

## 4. Análise por Categoria de Produto

> Volume de vendas alto não significa necessariamente receita alta — aqui separamos os dois.

In [ ]:
# Receita estimada = preço × quantidade
amazon_df['receita'] = amazon_df['price'] * amazon_df['quantity_sold']

cat_analise = amazon_df.groupby('product_category').agg(
    Total_Vendas=('quantity_sold', 'sum'),
    Receita_Total=('receita', 'sum'),
    Preco_Medio=('price', 'mean'),
    Qtd_Produtos=('price', 'count')
).round(2).sort_values('Receita_Total', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cores = sns.color_palette('muted', len(cat_analise))

# Volume vendido por categoria
vol_sorted = cat_analise['Total_Vendas'].sort_values()
axes[0].barh(vol_sorted.index, vol_sorted.values, color=cores)
axes[0].set_title('Volume Total Vendido por Categoria', fontweight='bold')
axes[0].set_xlabel('Quantidade Vendida')

# Receita por categoria
rec_sorted = cat_analise['Receita_Total'].sort_values()
bars = axes[1].barh(rec_sorted.index, rec_sorted.values, color=cores)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_title('Receita Total por Categoria (US$)', fontweight='bold')
axes[1].set_xlabel('Receita (US$)')

plt.tight_layout()
plt.show()

print('=== Tabela Completa por Categoria ===')
cat_analise

## 5. Distribuição Geográfica dos Clientes

> Saber de onde vêm os clientes orienta decisões de logística, marketing e expansão.

In [ ]:
regiao_analise = amazon_df.groupby('customer_region').agg(
    Total_Pedidos=('quantity_sold', 'count'),
    Quantidade_Vendida=('quantity_sold', 'sum'),
    Receita_Total=('receita', 'sum'),
    Ticket_Medio=('price', 'mean')
).round(2).sort_values('Receita_Total', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cores_r = sns.color_palette('Set2', len(regiao_analise))

# Pedidos por região
axes[0].bar(regiao_analise.index, regiao_analise['Total_Pedidos'], color=cores_r)
axes[0].set_title('Total de Pedidos por Região', fontweight='bold')
axes[0].set_xlabel('Região')
axes[0].set_ylabel('Pedidos')
axes[0].tick_params(axis='x', rotation=30)

# Pizza de receita por região
axes[1].pie(
    regiao_analise['Receita_Total'],
    labels=regiao_analise.index,
    autopct='%1.1f%%',
    colors=cores_r,
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[1].set_title('Participação na Receita por Região (%)', fontweight='bold')

plt.tight_layout()
plt.show()

print('=== Tabela por Região ===')
regiao_analise

## 6. Correlação: Preço × Quantidade Vendida

> Produtos mais caros vendem menos? Essa análise testa essa hipótese e revela o comportamento de demanda.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter preço x quantidade
categorias = amazon_df['product_category'].unique()
palette = dict(zip(categorias, sns.color_palette('muted', len(categorias))))

for cat in categorias:
    subset = amazon_df[amazon_df['product_category'] == cat]
    axes[0].scatter(subset['price'], subset['quantity_sold'],
                    label=cat, alpha=0.5, s=20, color=palette[cat])

axes[0].set_title('Preço × Quantidade Vendida por Categoria', fontweight='bold')
axes[0].set_xlabel('Preço (US$)')
axes[0].set_ylabel('Quantidade Vendida')
axes[0].legend(fontsize=7, loc='upper right')

# Preço médio por categoria — comparativo
preco_cat = amazon_df.groupby('product_category')['price'].mean().sort_values(ascending=True)
axes[1].barh(preco_cat.index, preco_cat.values,
             color=sns.color_palette('muted', len(preco_cat)))
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))
axes[1].set_title('Ticket Médio por Categoria (US$)', fontweight='bold')
axes[1].set_xlabel('Preço Médio (US$)')

plt.tight_layout()
plt.show()

corr = amazon_df['price'].corr(amazon_df['quantity_sold'])
print(f'Correlação de Pearson entre preço e quantidade vendida: {corr:.4f}')
if abs(corr) < 0.2:
    print('→ Correlação fraca: preço sozinho não explica o volume de vendas.')
elif corr < 0:
    print('→ Correlação negativa: produtos mais caros tendem a vender menos.')
else:
    print('→ Correlação positiva: produtos mais caros tendem a vender mais.')

## 7. Heatmap — Receita por Categoria e Região

> O cruzamento categoria × região revela onde cada tipo de produto tem mais tração.

In [ ]:
pivot = amazon_df.pivot_table(
    values='receita',
    index='customer_region',
    columns='product_category',
    aggfunc='sum'
).fillna(0).round(0)

plt.figure(figsize=(13, 6))
sns.heatmap(
    pivot,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Receita Total (US$)'}
)
plt.title('Receita Total por Categoria e Região (US$)', fontsize=14, fontweight='bold')
plt.xlabel('Categoria de Produto')
plt.ylabel('Região do Cliente')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Conclusões e Insights

### 💰 Preços
- A distribuição de preços é assimétrica à direita — a maioria dos produtos está em faixas mais acessíveis, com poucos itens premium puxando a média para cima. A mediana é um indicador mais confiável do preço típico do que a média.

### 📦 Categorias
- A categoria com maior volume vendido nem sempre é a de maior receita. Isso indica diferenças significativas no ticket médio entre categorias — oportunidade de revisão de mix e precificação.

### 🌎 Regiões
- A concentração de receita em poucas regiões sugere que há mercados subexplorados. Regiões com bom ticket médio mas baixo volume são candidatas a campanhas de aquisição.

### 📈 Correlação Preço x Volume
- A análise de correlação responde se existe uma relação clara de demanda — resultado interpretado automaticamente na célula acima com base nos dados reais.

### 📌 Próximos passos sugeridos
1. Analisar séries temporais de vendas (se houver coluna de data)
2. Criar score de valor por cliente (RFM) com dados de região e volume
3. Testar modelos de regressão para prever quantidade vendida com base no preço e categoria